In [1]:
from torch import nn


class VanillaSkipgram(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim
        )
        self.linear = nn.Linear(
            in_features=embedding_dim,
            out_features=vocab_size
        )

    def forward(self, input_ids):
        embeddings = self.embedding(input_ids)
        output = self.linear(embeddings)
        return output

import pandas as pd
from Korpora import Korpora
from konlpy.tag import Okt


corpus = Korpora.load("nsmc")
corpus = pd.DataFrame(corpus.test)

In [3]:
import pandas as pd
from Korpora import Korpora
from konlpy.tag import Okt


corpus = Korpora.load("nsmc")
corpus = pd.DataFrame(corpus.test)


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/



[nsmc] download ratings_train.txt: 14.6MB [00:00, 58.6MB/s]                            
[nsmc] download ratings_test.txt: 4.90MB [00:00, 49.6MB/s]


In [ ]:
tokenizer = Okt()   # Okt 토크나이저로 형태소 추출
tokens = [tokenizer.morphs(review) for review in corpus.text]
print(tokens[:3])

[['굳', 'ㅋ'], ['GDNTOPCLASSINTHECLUB'], ['뭐', '야', '이', '평점', '들', '은', '....', '나쁘진', '않지만', '10', '점', '짜', '리', '는', '더', '더욱', '아니잖아']]


In [ ]:
from collections import Counter


def build_vocab(corpus, n_vocab, special_tokens):
    """
    단어 사전을 구축하는 함수
    input:
        n_vocab: 구축할 단어 사전의 크기
        special_tokens : 특별한 의미를 갖는 토큰들(ex. <UNK>)
    output:
        생성한 단어 사전
    """
    counter = Counter()
    for tokens in corpus:
        counter.update(tokens)
    vocab = special_tokens
    for token, count in counter.most_common(n_vocab):
        vocab.append(token)
    return vocab

# vocab size는 5000개의 토큰과 <unk>토큰을 포함해서 5001개가 된다.
vocab = build_vocab(corpus=tokens, n_vocab=5000, special_tokens=["<unk>"])
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for idx, token in enumerate(vocab)}

print(vocab[:10])
print(len(vocab))

['<unk>', '.', '이', '영화', '의', '..', '가', '에', '...', '을']
5001


In [12]:
print(token_to_id)

{'<unk>': 0, '.': 1, '이': 2, '영화': 3, '의': 4, '..': 5, '가': 6, '에': 7, '...': 8, '을': 9, '도': 10, '들': 11, ',': 12, '는': 13, '를': 14, '은': 15, '너무': 16, '한': 17, '?': 18, '다': 19, '정말': 20, '만': 21, '진짜': 22, '적': 23, '!': 24, '로': 25, '점': 26, '으로': 27, '에서': 28, '평점': 29, '연기': 30, '것': 31, '과': 32, '~': 33, '최고': 34, '내': 35, '그': 36, '나': 37, '잘': 38, '인': 39, '와': 40, '안': 41, '생각': 42, '게': 43, '이런': 44, '못': 45, '왜': 46, '스토리': 47, '....': 48, '이다': 49, '드라마': 50, '사람': 51, '감동': 52, '하는': 53, '1': 54, '보고': 55, '때': 56, '더': 57, '하고': 58, '고': 59, '말': 60, '아': 61, '감독': 62, '배우': 63, 'ㅋㅋ': 64, '내용': 65, '그냥': 66, '거': 67, '중': 68, '까지': 69, '재미': 70, '보다': 71, '본': 72, '요': 73, '!!': 74, '없는': 75, '좀': 76, '뭐': 77, '시간': 78, '수': 79, '지': 80, '봤는데': 81, '쓰레기': 82, '사랑': 83, '볼': 84, '네': 85, '작품': 86, '다시': 87, '하나': 88, '없다': 89, '10': 90, '할': 91, '이건': 92, '마지막': 93, '2': 94, '저': 95, '같은': 96, '정도': 97, '있는': 98, 'ㅠㅠ': 99, 'ㅋ': 100, '좋은': 101, '완전': 102, '처음': 103, '대': 10

In [ ]:
def get_word_pairs(tokens, window_size):
    """
    skip-gram 모델의 입력데이터로 사용할 수 있도록 전처리 하는 함수
    input:
        window_size: 주변 단어를 몇개까지 고려할지
    output:
        중신 단어와 주변 단어를 고려한 쌍 생성
    """
    pairs = []
    for sentence in tokens:
        sentence_length = len(sentence)
        for idx, center_word in enumerate(sentence):
            window_start = max(0, idx - window_size)
            window_end = min(sentence_length, idx + window_size + 1)
            center_word = sentence[idx]
            context_words = sentence[window_start:idx] + sentence[idx+1:window_end]
            for context_word in context_words:
                pairs.append([center_word, context_word])
    return pairs


word_pairs = get_word_pairs(tokens, window_size=2)
print(word_pairs[:5])

[['굳', 'ㅋ'], ['ㅋ', '굳'], ['뭐', '야'], ['뭐', '이'], ['야', '뭐']]


In [ ]:
def get_index_pairs(word_pairs, token_to_id):
    """
    단어 쌍을 토큰 인덱스 쌍으로 변환하는 함수
    input:
        word_pairs: get_word_pairs()로 생성한 window_size에 따른 단어 쌍
        token_to_id: vocab 사전에서 추출한 단어 인덱스
    """
    pairs = []
    unk_index = token_to_id["<unk>"]
    for word_pair in word_pairs:
        center_word, context_word = word_pair
        center_index = token_to_id.get(center_word, unk_index)
        context_index = token_to_id.get(context_word, unk_index)
        pairs.append([center_index, context_index])
    return pairs

index_pairs = get_index_pairs(word_pairs, token_to_id)
print(index_pairs[:5])
print(len(vocab))

[[595, 100], [100, 595], [77, 176], [77, 2], [176, 77]]
5001


In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# index_pairs리스트를 텐서 형식으로 변환
index_pairs = torch.tensor(index_pairs)
center_indexes = index_pairs[:, 0]
context_indexes = index_pairs[:, 1]

dataset = TensorDataset(center_indexes, context_indexes)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [ ]:
from torch import optim


device = "cuda" if torch.cuda.is_available() else "cpu"
word2vec = VanillaSkipgram(vocab_size=len(token_to_id), embedding_dim=128).to(device)
# 클래스 분류 문제이므로 CrossEntropy 사용
criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.SGD(word2vec.parameters(), lr=0.1)

In [ ]:
for epoch in range(10):
    cost = 0.0
    for input_ids, target_ids in dataloader:
        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)

        logits = word2vec(input_ids)
        loss = criterion(logits, target_ids)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        cost += loss

    cost = cost / len(dataloader)   # 해당 epoch 평균손실
    print(f"Epoch : {epoch+1:4d}, Cost : {cost:.3f}")

Epoch :    1, Cost : 6.197
Epoch :    2, Cost : 5.982
Epoch :    3, Cost : 5.932
Epoch :    4, Cost : 5.902
Epoch :    5, Cost : 5.880
Epoch :    6, Cost : 5.862
Epoch :    7, Cost : 5.847
Epoch :    8, Cost : 5.834
Epoch :    9, Cost : 5.823
Epoch :   10, Cost : 5.812


In [ ]:
token_to_embedding = dict()
embedding_matrix = word2vec.embedding.weight.detach().cpu().numpy()

for word, embedding in zip(vocab, embedding_matrix):
    token_to_embedding[word] = embedding

index = 30
token = vocab[index]
token_embedding = token_to_embedding[token]
print(token)
print(token_embedding)

연기
[ 0.22378263 -0.39101735 -0.4826897  -0.15122272 -1.2413721  -0.97106737
  1.2847497  -0.23734558 -0.02817659  0.7545556  -1.9338048  -0.37275335
 -0.60969627  0.4825486  -0.1753931   1.3885697  -1.2604601  -0.5318901
 -0.57832986  0.44824207 -0.30895686 -1.2802241   0.2729923  -0.707793
 -0.1699813  -0.06842904 -0.40528083 -0.15523064  0.06126551  0.04830118
  0.04275406  0.2762147   0.5356206   0.88573444 -2.1593153   0.68071055
 -0.8986641   0.6407826   0.82842225 -1.00541    -0.85454094  1.1266356
  0.6418499  -1.0813174   0.13582875 -0.72374713  0.32893834 -0.87410015
  0.2940894   2.7551355   0.07333601  0.9445694  -0.49594417 -2.32964
  1.4279528  -0.21539707  1.0252205  -1.2519293   0.09192015 -0.5011765
  1.2230842   0.5787118   0.6421546  -0.01070618 -0.18877517  1.2042818
  0.92258537 -0.5536464   2.37848    -0.1566905   0.34592465 -1.3455489
  0.20558612  0.00776275 -0.4435568   0.56390405 -0.20905782  0.4709764
 -1.2988843   0.5621871   0.5143561   0.31996885  1.1243095

In [ ]:
import numpy as np
from numpy.linalg import norm


def cosine_similarity(a, b):
    # 코사인 유사도를 계산하여 단어 간의 유사도 측정
    cosine = np.dot(b, a) / (norm(b, axis=1) * norm(a))
    return cosine

def top_n_index(cosine_matrix, n):
    closest_indexes = cosine_matrix.argsort()[::-1]
    top_n = closest_indexes[1 : n + 1]
    return top_n


cosine_matrix = cosine_similarity(token_embedding, embedding_matrix)
top_n = top_n_index(cosine_matrix, n=5)

print(f"{token}와 가장 유사한 5 개 단어")
for index in top_n:
    print(f"{id_to_token[index]} - 유사도 : {cosine_matrix[index]:.4f}")

연기와 가장 유사한 5 개 단어
오빠 - 유사도 : 0.3899
널 - 유사도 : 0.2797
돼 - 유사도 : 0.2750
즐거운 - 유사도 : 0.2735
단순 - 유사도 : 0.2721
